# 03. SVM Training

## Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import time
import json
from tqdm.auto import tqdm

from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from scipy.stats import loguniform, uniform
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

np.random.seed(42)

print("[03-SVM-TRAINING] Libraries loaded")

## Configuration

In [ ]:
# Input paths (from 02-Feature-Extraction-Step)
INPUT_DIR = '/kaggle/input/02-feature-extraction-result'

# Fallback for local testing
if not os.path.exists(INPUT_DIR):
    INPUT_DIR = './02-feature-extraction-output'

# Output directory
OUTPUT_DIR = './03-svm-training-output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Class configuration
CLASSES = ['Organik', 'Anorganik', 'Lainnya']

CV_FOLDS = 5
N_ITER = 30

print(f"[03-SVM-TRAINING] Input directory: {INPUT_DIR}")
print(f"[03-SVM-TRAINING] Output directory: {OUTPUT_DIR}")
print(f"[03-SVM-TRAINING] CV folds: {CV_FOLDS}")
print(f"[03-SVM-TRAINING] Search iterations: {N_ITER}")

## Load Normalized Features from 02-Feature-Extraction

In [ ]:
print("[03-SVM-TRAINING] Loading normalized features...")

# Load features (ALREADY NORMALIZED!)
X_train = np.load(f'{INPUT_DIR}/train_mobilenet_features.npy')
X_val = np.load(f'{INPUT_DIR}/val_mobilenet_features.npy')
X_test = np.load(f'{INPUT_DIR}/test_mobilenet_features.npy')

# Load labels
y_train = np.load(f'{INPUT_DIR}/train_labels.npy')
y_val = np.load(f'{INPUT_DIR}/val_labels.npy')
y_test = np.load(f'{INPUT_DIR}/test_labels.npy')

print(f"[03-SVM-TRAINING] Data loaded:")
print(f"  Train: {X_train.shape} features, {len(y_train)} labels")
print(f"  Val:   {X_val.shape} features, {len(y_val)} labels")
print(f"  Test:  {X_test.shape} features, {len(y_test)} labels")

print(f"\n[03-SVM-TRAINING] Feature statistics:")
print(f"  Train mean: {X_train.mean():.6f} (should be ~0)")
print(f"  Train std:  {X_train.std():.6f} (should be ~1)")
print(f"  Val mean:   {X_val.mean():.6f}")
print(f"  Test mean:  {X_test.mean():.6f}")

## Verify Feature Normalization Quality

In [ ]:
print("[03-SVM-TRAINING] Verifying normalization quality...")

feature_means = X_train.mean(axis=0)
feature_stds = X_train.std(axis=0)

print(f"  Feature means - Mean: {feature_means.mean():.6f}, Std: {feature_means.std():.6f}")
print(f"  Feature stds  - Mean: {feature_stds.mean():.6f}, Std: {feature_stds.std():.6f}")
print(f"  Zero variance features: {(feature_stds == 0).sum()}")

if abs(feature_means.mean()) < 0.01 and abs(feature_stds.mean() - 1.0) < 0.1:
    print("\n[03-SVM-TRAINING] ✓ Features properly normalized (StandardScaler applied in 02)")
else:
    print("\n[03-SVM-TRAINING] ⚠ WARNING: Features may not be properly normalized!")

## SVM Training Functions

In [ ]:
def optimize_svm(X, y, kernel_type='rbf', n_iter=30):
    """
    Optimize SVM hyperparameters using RandomizedSearchCV.
    
    """
    print(f"\n[03-SVM-TRAINING] Optimizing {kernel_type.upper()} kernel...")
    print(f"  Dataset: {X.shape[0]:,} samples × {X.shape[1]:,} features")
    print(f"  Iterations: {n_iter}")
    print(f"  CV folds: {CV_FOLDS}")
    print(f"  Each iteration trains {CV_FOLDS} models (one per fold)")
    
    # Define parameter distributions
    if kernel_type == 'rbf':
        param_dist = {
            'C': [0.1, 1, 10, 100, 1000],
            'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
            'class_weight': [None, 'balanced']
        }
    else:  # poly
        param_dist = {
            'C': [0.1, 1, 10, 100, 1000],
            'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
            'degree': [2, 3, 4],
            'coef0': [0, 0.1, 0.5, 1],
            'class_weight': [None, 'balanced']
        }
    
    # Base SVM with probability=True
    svm = SVC(kernel=kernel_type, probability=True, random_state=42)
    
    # Stratified K-Fold CV
    cv_splitter = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    
    # RandomizedSearchCV
    random_search = RandomizedSearchCV(
        estimator=svm,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv_splitter,
        scoring='f1_weighted',
        n_jobs=-1,
        random_state=42,
        verbose=1
    )
    
    # Fit on FULL training set
    # RandomizedSearchCV will do internal CV splitting
    random_search.fit(X, y)
    
    best_params = random_search.best_params_
    best_score = random_search.best_score_
    
    print(f"\n[03-SVM-TRAINING] Optimization complete!")
    print(f"  Best CV score: {best_score:.4f}")
    print(f"  Best parameters:")
    for param, value in best_params.items():
        print(f"    {param}: {value}")
    
    return best_params, best_score


def train_final_svm(X_train, y_train, X_val, y_val, kernel_type, best_params):
    """
    Train final SVM with best hyperparameters on FULL training set.
    
    """
    print(f"\n[03-SVM-TRAINING] Training final {kernel_type.upper()} SVM...")
    print(f"  Training on: {len(X_train):,} samples")
    print(f"  Validating on: {len(X_val):,} samples")
    
    start_time = time.time()
    
    # CRITICAL: probability=True
    svm = SVC(kernel=kernel_type, probability=True, random_state=42, **best_params)
    svm.fit(X_train, y_train)
    
    train_time = time.time() - start_time
    
    # Validate
    y_val_pred = svm.predict(X_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')
    
    print(f"  Training time: {train_time:.2f}s")
    print(f"  Val accuracy: {val_accuracy:.4f}")
    print(f"  Val F1-score: {val_f1:.4f}")
    
    return svm, val_accuracy, val_f1, train_time

print("[03-SVM-TRAINING] Training functions loaded")
print("  ✓ optimize_svm: Hyperparameter optimization with full train set + CV")
print("  ✓ train_final_svm: Final model training and validation")

## Train SVM Models (RBF + Polynomial)

In [ ]:
print("[03-SVM-TRAINING] Starting SVM training...")
print("="*60)

print("\n[03-SVM-TRAINING] Hyperparameter optimization")
print(f"  Training samples: {len(X_train):,}")
print(f"  CV strategy: {CV_FOLDS}-fold StratifiedKFold")
print(f"  Each fold will use ~{int(len(X_train) * (CV_FOLDS-1)/CV_FOLDS):,} samples for training")
print(f"  Each fold will use ~{int(len(X_train) / CV_FOLDS):,} samples for validation")

# Train both kernels
kernels = ['rbf', 'poly']
models = {}
results = {}

total_start_time = time.time()

for kernel_type in kernels:
    print(f"\n{'='*60}")
    print(f"Training {kernel_type.upper()} kernel SVM")
    print(f"{'='*60}")
    
    best_params, best_cv_score = optimize_svm(
        X_train, y_train,
        kernel_type=kernel_type, 
        n_iter=N_ITER
    )
    
    model, val_acc, val_f1, train_time = train_final_svm(
        X_train, y_train, 
        X_val, y_val, 
        kernel_type, 
        best_params
    )
    
    models[kernel_type] = model
    results[kernel_type] = {
        'best_params': best_params,
        'best_cv_score': best_cv_score,
        'val_accuracy': val_acc,
        'val_f1': val_f1,
        'train_time': train_time
    }

total_time = time.time() - total_start_time

print(f"\n{'='*60}")
print(f"[03-SVM-TRAINING] Training Complete!")
print(f"{'='*60}")
print(f"Total training time: {total_time:.2f}s ({total_time/60:.1f} min)")
print(f"\nModels trained: {list(models.keys())}")
print(f"\nResults summary:")
for kernel, res in results.items():
    print(f"\n  {kernel.upper()} kernel:")
    print(f"    Best CV score: {res['best_cv_score']:.4f}")
    print(f"    Val accuracy: {res['val_accuracy']:.4f}")
    print(f"    Val F1: {res['val_f1']:.4f}")
    print(f"    Training time: {res['train_time']:.2f}s")

## Model Comparison & Selection

In [7]:
print("\n[03-SVM-TRAINING] Model Comparison:")
print("="*60)

comparison_data = []
for kernel_type, result in results.items():
    comparison_data.append({
        'Kernel': kernel_type.upper(),
        'CV Score': result['best_cv_score'],
        'Val Accuracy': result['val_accuracy'],
        'Val F1': result['val_f1'],
        'Train Time (s)': result['train_time']
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

# Select best model
best_kernel = max(results.keys(), key=lambda k: results[k]['val_accuracy'])
best_model = models[best_kernel]
best_result = results[best_kernel]

print(f"\n[03-SVM-TRAINING] Best model: {best_kernel.upper()}")
print(f"  Val accuracy: {best_result['val_accuracy']:.4f}")
print(f"  Val F1-score: {best_result['val_f1']:.4f}")


[03-SVM-TRAINING] Model Comparison:
Kernel  CV Score  Val Accuracy   Val F1  Train Time (s)
   RBF  0.977378      0.984074 0.984076      177.480335
  POLY  0.980240      0.985926 0.985936      242.249705

[03-SVM-TRAINING] Best model: POLY
  Val accuracy: 0.9859
  Val F1-score: 0.9859


## Save Trained Models

In [ ]:
print(f"\n[03-SVM-TRAINING] Saving models to: {OUTPUT_DIR}")

# Save both models
for kernel_type, model in models.items():
    model_filename = f'{OUTPUT_DIR}/MobileNetV3_{kernel_type}_model.pkl'
    joblib.dump(model, model_filename)
    
    model_size = os.path.getsize(model_filename) / (1024 * 1024)
    print(f"  ✓ MobileNetV3_{kernel_type}_model.pkl ({model_size:.2f} MB)")

# Save training results
results_to_save = {
    kernel: {
        'best_params': {k: str(v) if not isinstance(v, (int, float, str)) else v 
                       for k, v in result['best_params'].items()},
        'best_cv_score': float(result['best_cv_score']),
        'val_accuracy': float(result['val_accuracy']),
        'val_f1': float(result['val_f1']),
        'train_time': float(result['train_time'])
    }
    for kernel, result in results.items()
}

with open(f'{OUTPUT_DIR}/training_results.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)

print(f"  ✓ training_results.json")

# Save comparison table
df_comparison.to_csv(f'{OUTPUT_DIR}/model_comparison.csv', index=False)
print(f"  ✓ model_comparison.csv")

print(f"\n[03-SVM-TRAINING] All models saved")

## Test Probability Estimation

In [ ]:
print("[03-SVM-TRAINING] Testing probability estimation...")
print("="*60)

# Test on first 10 validation samples
test_samples = X_val[:10]
test_labels = y_val[:10]

predictions = best_model.predict(test_samples)
probabilities = best_model.predict_proba(test_samples)

print(f"\nSample Predictions ({best_kernel.upper()} kernel):")
print("-" * 80)
print(f"{'True':<12} {'Predicted':<12} {'Confidence':<12} {'Probabilities':<45}")
print("-" * 80)

for i in range(len(test_samples)):
    true_label = CLASSES[test_labels[i]]
    pred_label = CLASSES[predictions[i]]
    confidence = probabilities[i].max()
    probs_str = ' | '.join([f"{CLASSES[j]}: {probabilities[i][j]:.3f}" for j in range(3)])
    
    print(f"{true_label:<12} {pred_label:<12} {confidence:<12.3f} {probs_str}")

print("-" * 80)

# Confidence statistics
all_val_probs = best_model.predict_proba(X_val)
all_confidences = all_val_probs.max(axis=1)

print(f"\n[03-SVM-TRAINING] Confidence distribution:")
print(f"  Min:  {all_confidences.min():.3f}")
print(f"  Max:  {all_confidences.max():.3f}")
print(f"  Mean: {all_confidences.mean():.3f}")
print(f"  Std:  {all_confidences.std():.3f}")

if all_confidences.std() > 0.05:
    print(f"\n[03-SVM-TRAINING] ✓ Confidence varies properly (not stuck at constant value!)")
else:
    print(f"\n[03-SVM-TRAINING] WARNING: Confidence has low variability")

## Training Summary

In [ ]:
summary = f"""
========================================
03. SVM TRAINING SUMMARY - JakOlah
========================================

Training Strategy:
  1. Hyperparameter Optimization:
     - Use FULL training set ({len(X_train):,} samples)
     - RandomizedSearchCV does internal CV splitting
     - Each fold: ~{int(len(X_train) * 2/3):,} train, ~{int(len(X_train) / 3):,} val
     - Best params selected by averaging across {CV_FOLDS} folds
  
  2. Final Model Training:
     - Train on FULL training set with best params
     - Validate on separate val set ({len(X_val):,} samples = 15% of total)

Kernels Trained:
"""

for kernel_type, result in results.items():
    summary += f"""
  {kernel_type.upper()} Kernel:
    Best CV Score (F1): {result['best_cv_score']:.4f} (from hyperparameter search)
    Val Accuracy: {result['val_accuracy']:.4f} ({result['val_accuracy']*100:.2f}%)
    Val F1-Score: {result['val_f1']:.4f}
    Training Time: {result['train_time']:.1f}s
    Best Params: {result['best_params']}
"""

summary += f"""
Best Model:
  Kernel: {best_kernel.upper()}
  Val Accuracy: {best_result['val_accuracy']:.4f}
  Val F1-Score: {best_result['val_f1']:.4f}

Output Files:
  ✓ MobileNetV3_rbf_model.pkl
  ✓ MobileNetV3_poly_model.pkl
  ✓ training_results.json
  ✓ model_comparison.csv

========================================
NEXT STEP: Run 04-Evaluation-Step.ipynb
========================================
"""

print(summary)

with open(f'{OUTPUT_DIR}/training_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary)

print(f"[03-SVM-TRAINING] Summary saved to {OUTPUT_DIR}/training_summary.md")
print(f"[03-SVM-TRAINING] ✅ COMPLETED")

## Download Output (Kaggle)

In [ ]:
import shutil

# Create zip file of all outputs
zip_filename = '03-svm-training-output'
shutil.make_archive(zip_filename, 'zip', OUTPUT_DIR)

print(f"[03-SVM-TRAINING] Output zipped to: {zip_filename}.zip")
print(f"  File size: {os.path.getsize(f'{zip_filename}.zip') / (1024*1024):.2f} MB")
print(f"\n💾 Download {zip_filename}.zip dari Kaggle output panel")